# Módulo Especial de Consultoria na Área de Dados com Agentes de IA

## Projeto Prático: Day Trade Analytics em Tempo Real com Agentes de IA

**Data Science Academy** — versão modernizada para Python 3.12 + `uv` + VSCode/WSL.

---

Este capítulo é diferente dos anteriores: o material original não é um notebook,
é um **aplicativo web** (`dsa_app.py`) feito com Streamlit. Este notebook existe
para você **entender e testar cada peça separadamente** antes de rodar o app
inteiro — é muito mais fácil descobrir por que algo não funciona olhando uma
peça por vez do que olhando o app já montado.

O caminho é este:

| Parte | O que você faz |
|---|---|
| 1. Setup | Prepara o ambiente (uv, Ollama, kernel do VSCode) |
| 2. As peças | Testa dados, gráficos, ferramentas e agentes, um a um |
| 3. O app | Sobe o `dsa_app.py` com tudo funcionando |
| 4. Trocando o motor | Roda o mesmo código com Groq, OpenAI, Anthropic, Grok e Ollama |
| 5. Rotina e problemas | O dia a dia e a tabela de "deu erro, e agora?" |

> **O que mudou em relação ao vídeo do curso.** O material foi montado em
> fevereiro de 2025. Três coisas quebraram desde então, e todas estão
> corrigidas aqui, com o código original preservado comentado para você comparar:
>
> 1. os modelos `deepseek-r1-distill-llama-70b` e `llama-3.3-70b-versatile`
>    **foram descontinuados pela Groq** e hoje retornam erro 404;
> 2. o pacote `duckduckgo_search` **foi renomeado** para `ddgs` — o nome antigo
>    ainda instala, mas devolve busca vazia sem avisar;
> 3. o `load_dotenv()` sem argumento **procura o `.env` ao lado do script**, o
>    que faz o app reclamar de chave ausente dependendo da pasta de onde você o inicia.

---
# Parte 1 — Setup do ambiente

## 1.1 Instalar o `uv`

O `uv` é o substituto moderno do `conda` deste material. A diferença prática:
o `conda` instala um Python gigante e compartilhado entre projetos; o `uv` cria
uma pasta `.venv` **dentro do capítulo**, com as versões exatas que ele precisa.

Pense assim: o `conda` é um armário de ferramentas comunitário do prédio inteiro —
alguém troca uma chave de lugar e o seu projeto quebra. O `uv` é uma caixinha de
ferramentas por projeto. Nada vaza de um para o outro.

Ele também é **muito** mais rápido: o que o `pip` faz em minutos, o `uv` faz em segundos.

As células abaixo têm os comandos **comentados** de propósito — eles são de
terminal, não de Python. Copie a linha, cole no terminal e rode.

In [ ]:
# ============================================================================
# INSTALAR O uv  (rode no TERMINAL, nao aqui)
# ============================================================================

# --- Linux, WSL ou macOS ---
# curl -LsSf https://astral.sh/uv/install.sh | sh

# Depois de instalar, feche e reabra o terminal (ou rode a linha abaixo) para
# que o sistema encontre o comando novo:
# source $HOME/.local/bin/env

# --- Windows (PowerShell) ---
# powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# --- Conferir se deu certo ---
# uv --version

## 1.2 Criar o ambiente do capítulo

A pasta do capítulo já vem com um `pyproject.toml`, que é a lista de
ingredientes do projeto. Com ele, criar o ambiente é **um comando só**.

Uma observação que vale ouro: no `pyproject.toml` as versões estão escritas com
`>=` (por exemplo, `yfinance>=1.7`), e não com `==` como no `requirements.txt`
original. Travar tudo com `==` é como tirar uma foto: no dia seguinte já está
velha. O `>=` diz "desta versão para cima", e quem guarda a foto exata do que foi
instalado é o arquivo `uv.lock`, gerado automaticamente. Assim você tem
reprodutibilidade **sem** ficar preso a pacotes de dois anos atrás.

In [ ]:
# ============================================================================
# CRIAR O AMBIENTE  (rode no TERMINAL, dentro da pasta do capitulo)
# ============================================================================

# Le o pyproject.toml, baixa o Python 3.12 se precisar, cria a .venv e instala
# tudo. E o unico comando necessario:
# uv sync

# --- Se voce quisesse comecar um projeto do ZERO (nao e o caso aqui) ---
# uv init --bare --python 3.12
# uv add streamlit yfinance plotly phidata groq openai anthropic ollama ddgs python-dotenv

# --- Comandos uteis do dia a dia ---
# uv run streamlit run dsa_app.py    # roda o app usando a .venv
# uv run python dsa_lista_modelos.py # roda um script usando a .venv
# uv add nome-do-pacote              # acrescenta uma dependencia nova
# uv sync --upgrade                  # atualiza tudo dentro dos limites do pyproject

## 1.3 Instalar o LLM local (Ollama)

O capítulo original usa a **Groq**, que é um serviço de nuvem e exige chave de
API. O Ollama é a alternativa que roda **na sua própria máquina**: não pede
chave, não cobra nada e funciona sem internet. Ele é a rede de segurança deste
notebook — se você não tiver nenhuma chave, tudo aqui continua funcionando.

> **Pré-requisito que derruba muita gente no WSL:** o instalador do Ollama
> descompacta os arquivos com `zstd`. Se o `zstd` não estiver instalado, a
> instalação morre com a mensagem
> `ERROR: This version requires zstd for extraction`.
> Instale o `zstd` **antes**.

Sobre o systemd: se `ps -p 1 -o comm=` responder `systemd` (é o caso da maioria
dos WSL modernos), o instalador registra o Ollama como serviço e ele sobe
sozinho no boot — você **não** precisa rodar `ollama serve` na mão.

In [ ]:
# ============================================================================
# INSTALAR O OLLAMA  (rode no TERMINAL, nao aqui)
# ============================================================================

# --- PASSO 1: o pre-requisito que quase ninguem lembra ---
# Sem isto o instalador falha com "ERROR: This version requires zstd for extraction"
# sudo apt-get update && sudo apt-get install -y zstd

# --- PASSO 2: instalar o Ollama (Linux / WSL) ---
# curl -fsSL https://ollama.com/install.sh | sh

# No Windows, baixe o instalador em: https://ollama.com/download

# --- PASSO 3: baixar o modelo ---
# ollama pull llama3.2

# --- PASSO 4: conferir ---
# ollama list

# --- O servidor esta no ar? ---
# Se o systemd estiver ativo (ps -p 1 -o comm= responde "systemd"), o Ollama ja
# sobe sozinho como servico e voce NAO precisa do comando abaixo.
# Se nao estiver, rode em outro terminal e deixe aberto:
# ollama serve

## 1.4 Registrar o kernel e selecionar no VSCode

"Kernel" é o Python que executa as células. Você precisa dizer ao VSCode para
usar **o Python da `.venv` deste capítulo**, e não o Python do sistema — que não
tem nenhuma das bibliotecas instaladas.

Depois de rodar o comando abaixo, no VSCode: **Select Kernel** (canto superior
direito) → **Python Environments...** → escolha `.venv (Python 3.12)`.

> Se o botão "Select Kernel" nem aparecer, faltam as extensões. Confira com
> `code --list-extensions` **dentro do WSL** e instale
> `ms-python.python` e `ms-toolsai.jupyter` — uma de cada vez, porque as duas no
> mesmo comando falham por concorrência.

In [ ]:
# ============================================================================
# REGISTRAR O KERNEL  (rode no TERMINAL, dentro da pasta do capitulo)
# ============================================================================

# uv run python -m ipykernel install --user \
#     --name dsa-12-projeto-daytrade \
#     --display-name "Python (uv) - DSA 12 Projeto Day Trade"

# --- Conferir as extensoes do VSCode DENTRO do WSL ---
# code --list-extensions

# --- Se faltar alguma, instale UMA DE CADA VEZ ---
# code --install-extension ms-python.python
# code --install-extension ms-toolsai.jupyter

# Extensoes instaladas com o VSCode aberto so valem apos Reload Window:
# Ctrl+Shift+P > "Developer: Reload Window"

## 1.5 Verificação do ambiente

**Esta célula você executa de verdade.** Ela responde à pergunta mais importante
antes de qualquer outra coisa: *o notebook está mesmo usando a `.venv` deste
capítulo?*

Olhe o `sys.executable`. Ele **tem que** terminar em
`12-Projeto/.venv/bin/python`. Se aparecer `/usr/bin/python3` ou algum caminho
com `anaconda3`, o kernel está errado e nada aqui vai funcionar — volte ao
passo 1.4.

In [ ]:
import sys
import platform
from importlib.metadata import version, PackageNotFoundError

print("Python :", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())
print()
print("sys.executable:")
print(" ", sys.executable)
print()

if ".venv" in sys.executable and "12-Projeto" in sys.executable:
    print("  >> CERTO: o notebook esta usando a .venv deste capitulo.")
else:
    print("  >> ATENCAO: kernel errado! Deveria terminar em 12-Projeto/.venv/bin/python")
    print("     Va em Select Kernel > Python Environments... > .venv (Python 3.12)")

print()
print("Versoes instaladas:")
for pacote in ["phidata", "groq", "openai", "anthropic", "ollama", "ddgs",
               "yfinance", "pandas", "plotly", "streamlit", "python-dotenv"]:
    try:
        print(f"  {pacote:<16} {version(pacote)}")
    except PackageNotFoundError:
        print(f"  {pacote:<16} NAO INSTALADO  (rode: uv sync)")

## 1.6 Conexão com o LLM local

**Célula executável.** Ela procura sozinha onde o Ollama está.

Por que "procurar"? Porque no WSL2 existe uma pegadinha clássica: você pode ter
instalado o Ollama **no Windows**, não no Linux. Nesse caso ele não está em
`localhost` do ponto de vista do WSL — o Windows é uma máquina vizinha na rede.
O endereço dele é o mesmo que aparece na linha `nameserver` do arquivo
`/etc/resolv.conf`.

A célula testa os dois endereços e diz qual funcionou. Se nenhum responder, ela
explica o que fazer — e o resto do notebook continua funcionando via nuvem.

In [ ]:
import os
import re
from pathlib import Path

import ollama

def dsa_descobre_ollama():
    # Candidatos, em ordem: o que estiver no .env, o local, e o Windows (WSL2).
    candidatos = []
    if os.getenv("OLLAMA_HOST"):
        candidatos.append(os.getenv("OLLAMA_HOST"))
    candidatos.append("http://localhost:11434")

    # No WSL2, o IP do Windows sai do nameserver do /etc/resolv.conf
    try:
        resolv = Path("/etc/resolv.conf").read_text()
        achado = re.search(r"nameserver\s+([0-9.]+)", resolv)
        if achado:
            candidatos.append(f"http://{achado.group(1)}:11434")
    except Exception:
        pass

    for endereco in candidatos:
        try:
            modelos = ollama.Client(host=endereco, timeout=5).list()["models"]
            return endereco, [m["model"] for m in modelos]
        except Exception:
            continue
    return None, []


OLLAMA_HOST, MODELOS_LOCAIS = dsa_descobre_ollama()

if OLLAMA_HOST:
    os.environ["OLLAMA_HOST"] = OLLAMA_HOST
    print(f"Ollama encontrado em: {OLLAMA_HOST}")
    print(f"Modelos baixados    : {MODELOS_LOCAIS or 'NENHUM'}")
    if not MODELOS_LOCAIS:
        print("\n  Nenhum modelo baixado. Rode no terminal:  ollama pull llama3.2")
    else:
        # Teste real de conversa
        cliente = ollama.Client(host=OLLAMA_HOST, timeout=120)
        modelo = os.getenv("OLLAMA_MODEL") or MODELOS_LOCAIS[0].split(":")[0]
        r = cliente.chat(model=modelo,
                         messages=[{"role": "user", "content": "Responda apenas: OK"}])
        # A partir da versao 0.4 o retorno e um ChatResponse, que aceita as duas
        # formas de acesso: por chave (como dicionario) e por atributo.
        print(f"\nTeste de conversa com '{modelo}':")
        print("  r['message']['content'] ->", r["message"]["content"].strip())
        print("  r.message.content       ->", r.message.content.strip())
else:
    print("Ollama NAO encontrado.")
    print("Enderecos testados: localhost e o IP do Windows (nameserver do /etc/resolv.conf)")
    print()
    print("Para instalar (no terminal):")
    print("  sudo apt-get install -y zstd")
    print("  curl -fsSL https://ollama.com/install.sh | sh")
    print("  ollama pull llama3.2")
    print()
    print("Sem problema: se voce tiver chave de API no .env, o resto funciona pela nuvem.")

---
# Parte 2 — As peças do app, uma a uma

Agora vamos abrir o `dsa_app.py` e testar cada pedaço isoladamente.

## 2.1 Chaves de API e o seletor de provedor

O app precisa de um LLM. O código original tinha o provedor e o modelo escritos
direto no meio do código. A versão modernizada olha o `.env`, descobre quais
chaves existem **de verdade** e usa a primeira que encontrar — caindo no Ollama
local se não houver nenhuma.

Repare no detalhe do `load_dotenv`: passamos o caminho explícito. Sem isso, ele
procura o `.env` a partir da pasta do arquivo que chamou, e você recebe um
"API key not set" enganoso mesmo com o `.env` preenchido corretamente.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# A pasta do capitulo. Num notebook nao existe __file__, entao usamos o
# diretorio de trabalho -- que o .vscode/settings.json ja fixa na pasta certa.
PASTA = Path.cwd()
load_dotenv(PASTA / ".env")

# Comecos de texto que significam "ainda nao preenchi esta chave"
DSA_PLACEHOLDERS = ("seu-", "sua-", "cole-", "coloque", "your-", "xxx", "<")


def dsa_chave_valida(nome_da_variavel):
    valor = (os.getenv(nome_da_variavel) or "").strip().strip('"').strip("'")
    if not valor or valor.lower().startswith(DSA_PLACEHOLDERS):
        return None
    return valor


DSA_PROVEDORES = [
    ("Groq",      "GROQ_API_KEY",      "GROQ_MODEL",      "openai/gpt-oss-120b"),
    ("OpenAI",    "OPENAI_API_KEY",    "OPENAI_MODEL",    "gpt-4o-mini"),
    ("Anthropic", "ANTHROPIC_API_KEY", "ANTHROPIC_MODEL", "claude-sonnet-4-5"),
    ("Grok",      "XAI_API_KEY",       "XAI_MODEL",       "grok-4"),
]

print("Chaves encontradas no .env:")
for apelido, var_chave, _, _ in DSA_PROVEDORES:
    marca = "CONFIGURADA" if dsa_chave_valida(var_chave) else "nao configurada"
    print(f"  {apelido:<10} {var_chave:<20} {marca}")


def dsa_detecta_provedor():
    for apelido, var_chave, var_modelo, padrao in DSA_PROVEDORES:
        chave = dsa_chave_valida(var_chave)
        if chave:
            return apelido, chave, os.getenv(var_modelo) or padrao
    return "Ollama", None, os.getenv("OLLAMA_MODEL") or "llama3.2"


DSA_PROVEDOR, DSA_CHAVE, DSA_MODELO = dsa_detecta_provedor()
print(f"\n>> Provedor escolhido: {DSA_PROVEDOR}  |  modelo: {DSA_MODELO}")

## 2.2 Os dados: Yahoo Finance

Esta é a parte do app que busca o histórico de preços. É a mesma função do
`dsa_app.py`, sem o `@st.cache_data` (que só faz sentido dentro do Streamlit).

Um detalhe importante que vale para o app inteiro: quando o ticker **não
existe**, o `yfinance` não levanta erro — ele devolve uma tabela **vazia**. Se
você não checar isso, o app desenha quatro gráficos em branco e ainda gasta uma
chamada paga de LLM perguntando sobre uma ação que não existe.

In [ ]:
import yfinance as yf

def dsa_extrai_dados(ticker, period="6mo"):
    stock = yf.Ticker(ticker)
    hist = stock.history(period=period)
    hist.reset_index(inplace=True)
    return hist


hist = dsa_extrai_dados("MSFT")
print("Linhas x colunas:", hist.shape)
print("Colunas:", list(hist.columns))
print()
display(hist.tail(3))

# E com um ticker que nao existe?
vazio = dsa_extrai_dados("ZZZZINVALIDO")
print(f"\nTicker inexistente -> tabela vazia? {vazio.empty}  (shape={vazio.shape})")
print("Por isso o app modernizado tem um 'if hist.empty: st.error(...); st.stop()'")

## 2.3 Os quatro gráficos

São as quatro funções de visualização do app. No app elas terminam com
`st.plotly_chart(fig, width="stretch")`; aqui trocamos por `fig.show()` para
desenhar dentro do notebook.

Sobre o `width="stretch"`: a página do app usa `layout="wide"`, mas
`st.plotly_chart(fig)` desenha o gráfico no tamanho nativo, deixando sobra dos
dois lados. O parâmetro antigo para corrigir isso era `use_container_width=True`,
hoje **deprecated** no Streamlit 1.63 — a forma atual é `width="stretch"`.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

ticker = "MSFT"

# --- 1. Preco de fechamento ---
fig = px.line(hist, x="Date", y="Close",
              title=f"{ticker} Preços das Ações (Últimos 6 Meses)", markers=True)
fig.show()

# --- 2. Candlestick ---
fig = go.Figure(data=[go.Candlestick(x=hist["Date"], open=hist["Open"],
                                     high=hist["High"], low=hist["Low"],
                                     close=hist["Close"])])
fig.update_layout(title=f"{ticker} Candlestick Chart (Últimos 6 Meses)")
fig.show()

# --- 3. Medias moveis ---
# SMA = media simples dos ultimos 20 dias (todos pesam igual).
# EMA = media exponencial: os dias recentes pesam mais, entao ela reage mais rapido.
h = hist.copy()
h["SMA_20"] = h["Close"].rolling(window=20).mean()
h["EMA_20"] = h["Close"].ewm(span=20, adjust=False).mean()
fig = px.line(h, x="Date", y=["Close", "SMA_20", "EMA_20"],
              title=f"{ticker} Médias Móveis (Últimos 6 Meses)",
              labels={"value": "Price (USD)", "Date": "Date"})
fig.show()

# --- 4. Volume ---
fig = px.bar(hist, x="Date", y="Volume",
             title=f"{ticker} Trading Volume (Últimos 6 Meses)")
fig.show()

## 2.4 A ferramenta de busca na web — e a quebra silenciosa

Aqui está a falha mais traiçoeira deste capítulo.

O código original usa `from phi.tools.duckduckgo import DuckDuckGo`. Por dentro,
essa ferramenta importa o pacote `duckduckgo_search`. Esse pacote **foi
renomeado** para `ddgs`. O nome antigo continua instalável, mas está abandonado:
emite um aviso de renomeação e, na esmagadora maioria das vezes, devolve **lista
vazia**. (Ele é instável, não morto: uma vez ou outra ainda responde. Isso é pior
do que estar quebrado de vez, porque esconde o problema.)

Ou seja: o agente de busca **não quebra com erro**. Ele simplesmente nunca acha
nada. É como mandar alguém pesquisar numa biblioteca que fechou: a pessoa volta
de mãos vazias e você não fica sabendo o motivo.

A célula abaixo demonstra as duas na prática — e depois monta a ferramenta nova.

In [ ]:
import json
import warnings

# ==============================================================================
# CODIGO ORIGINAL (mantido comentado para referencia)
# ==============================================================================
# from phi.tools.duckduckgo import DuckDuckGo
# ferramenta = DuckDuckGo()
# resultado = ferramenta.duckduckgo_search("Microsoft stock", max_results=3)
#
# ==============================================================================
# VERSAO MODERNA (Python 3.12 + uv)
# O que mudou: o pacote 'duckduckgo_search' foi renomeado para 'ddgs' e o nome
#              antigo hoje devolve zero resultados sem erro. A prova esta abaixo.
# ==============================================================================

print("=== Pacote ANTIGO (duckduckgo_search), que o phidata usa por dentro ===")
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    from duckduckgo_search import DDGS as DDGS_ANTIGO
    antigo = list(DDGS_ANTIGO().text("Microsoft stock", max_results=3))
print(f"  resultados: {len(antigo)}")
for a in avisos:
    print(f"  aviso: {a.category.__name__}: {a.message}")

print("\n=== Pacote NOVO (ddgs) ===")
from ddgs import DDGS
novo = list(DDGS().text("Microsoft stock", max_results=3))
print(f"  resultados: {len(novo)}")
if novo:
    print(f"  chaves de cada resultado: {list(novo[0].keys())}")
    print(f"  primeiro titulo: {novo[0]['title'][:70]}")

print()
print("=== O QUE VOCE ACABOU DE VER ===")
if len(antigo) == 0 and len(novo) > 0:
    print("  O pacote antigo devolveu ZERO e o novo devolveu resultados.")
    print("  Este e o caso normal: o agente de busca do capitulo ficava mudo.")
elif len(antigo) > 0:
    print(f"  Desta vez o pacote antigo respondeu ({len(antigo)} resultados).")
    print("  Ele e INSTAVEL, nao morto -- as vezes responde, quase sempre nao.")
    print("  Rode esta celula de novo algumas vezes e voce vera o zero aparecer.")
else:
    print("  Os dois devolveram zero: provavelmente a busca esta indisponivel")
    print("  agora (rede ou bloqueio temporario do DuckDuckGo). Tente mais tarde.")
print()
print("  O PONTO IMPORTANTE nao e o numero: e que o pacote antigo falha")
print("  SEM LEVANTAR ERRO. Um agente que nunca acha nada parece funcionar.")

Agora recriamos a ferramenta usando o pacote certo.

Repare que os **nomes dos métodos e o texto das docstrings são iguais** aos do
phidata. Isso não é enfeite: é a docstring que o phidata lê para explicar ao LLM
o que a ferramenta faz e quais argumentos ela aceita. Mudar o texto muda o
comportamento do agente.

In [ ]:
from phi.tools import Toolkit

class DSADuckDuckGo(Toolkit):
    """Busca na web usando o pacote ddgs (nome atual do duckduckgo_search)."""

    def __init__(self, fixed_max_results=5):
        super().__init__(name="duckduckgo")
        self.fixed_max_results = fixed_max_results
        self.register(self.duckduckgo_search)
        self.register(self.duckduckgo_news)

    def duckduckgo_search(self, query: str, max_results: int = 5) -> str:
        """Use this function to search DuckDuckGo for a query.

        Args:
            query(str): The query to search for.
            max_results (optional, default=5): The maximum number of results to return.

        Returns:
            The result from DuckDuckGo.
        """
        quantos = self.fixed_max_results or max_results
        try:
            return json.dumps(list(DDGS().text(query, max_results=quantos)), ensure_ascii=False)
        except Exception as erro:
            return json.dumps({"erro": f"busca indisponivel: {erro}"}, ensure_ascii=False)

    def duckduckgo_news(self, query: str, max_results: int = 5) -> str:
        """Use this function to get the latest news from DuckDuckGo.

        Args:
            query(str): The query to search for.
            max_results (optional, default=5): The maximum number of results to return.

        Returns:
            The latest news from DuckDuckGo.
        """
        quantos = self.fixed_max_results or max_results
        try:
            return json.dumps(list(DDGS().news(query, max_results=quantos)), ensure_ascii=False)
        except Exception as erro:
            return json.dumps({"erro": f"busca indisponivel: {erro}"}, ensure_ascii=False)


ferramenta = DSADuckDuckGo()
resultado = json.loads(ferramenta.duckduckgo_search("Microsoft MSFT stock", max_results=3))
print(f"Ferramenta nova devolveu {len(resultado)} resultados:")
for r in resultado:
    print(f"  - {r['title'][:70]}")

## 2.5 Os agentes

Chegamos ao coração do capítulo. São três agentes:

- **Agente de busca** — vasculha a web e sempre cita as fontes;
- **Agente financeiro** — consulta cotação, recomendações de analistas,
  fundamentos e notícias, via `YFinanceTools`;
- **Time** — o coordenador. Ele não faz o trabalho: ele **decide qual
  especialista chamar** e junta as respostas. É a diferença entre um funcionário
  e um gerente.

**A mudança mais importante deste capítulo está aqui.** Os IDs de modelo escritos
no código original foram descontinuados pela Groq — as duas chamadas hoje
retornam erro 404. Você pode confirmar isso rodando
`uv run python dsa_lista_modelos.py`.

E note o que **não** mudou: papéis, instruções, ferramentas, a montagem do time.
Só o objeto passado em `model=`. Guarde essa ideia, porque a Parte 4 é inteira
sobre ela.

In [ ]:
from phi.agent import Agent
from phi.tools.yfinance import YFinanceTools
from phi.model.groq import Groq
from phi.model.openai import OpenAIChat
from phi.model.anthropic import Claude
from phi.model.xai import xAI
from phi.model.ollama import Ollama


def dsa_cria_modelo():
    """Fabrica um objeto de LLM novo para o provedor detectado em 2.1.

    Por que uma funcao e nao uma variavel? Porque no phidata o objeto de modelo
    guarda estado (as ferramentas registradas naquele agente). Se os tres
    agentes dividissem o mesmo objeto, as ferramentas de um vazariam para o
    outro. A fabrica entrega um objeto limpo para cada um -- que e exatamente o
    que o codigo original fazia ao escrever Groq(...) tres vezes seguidas.
    """
    if DSA_PROVEDOR == "Groq":
        return Groq(id=DSA_MODELO, api_key=DSA_CHAVE)
    if DSA_PROVEDOR == "OpenAI":
        return OpenAIChat(id=DSA_MODELO, api_key=DSA_CHAVE)
    if DSA_PROVEDOR == "Anthropic":
        return Claude(id=DSA_MODELO, api_key=DSA_CHAVE)
    if DSA_PROVEDOR == "Grok":
        return xAI(id=DSA_MODELO, api_key=DSA_CHAVE)
    return Ollama(id=DSA_MODELO, host=os.getenv("OLLAMA_HOST") or "http://localhost:11434")


# ==============================================================================
# CODIGO ORIGINAL (mantido comentado para referencia)
# ==============================================================================
# dsa_agente_web_search = Agent(name="DSA Agente Web Search",
#                               role="Fazer busca na web",
#                               model=Groq(id="deepseek-r1-distill-llama-70b"),
#                               tools=[DuckDuckGo()],
#                               instructions=["Sempre inclua as fontes"],
#                               show_tool_calls=True, markdown=True)
#
# dsa_agente_financeiro = Agent(name="DSA Agente Financeiro",
#                               model=Groq(id="deepseek-r1-distill-llama-70b"),
#                               tools=[YFinanceTools(stock_price=True,
#                                                    analyst_recommendations=True,
#                                                    stock_fundamentals=True,
#                                                    company_news=True)],
#                               instructions=["Use tabelas para mostrar os dados"],
#                               show_tool_calls=True, markdown=True)
#
# multi_ai_agent = Agent(team=[dsa_agente_web_search, dsa_agente_financeiro],
#                        model=Groq(id="llama-3.3-70b-versatile"),
#                        instructions=["Sempre inclua as fontes", "Use tabelas para mostrar os dados"],
#                        show_tool_calls=True, markdown=True)
#
# ==============================================================================
# VERSAO MODERNA (Python 3.12 + uv)
# O que mudou: os modelos fixos ("deepseek-r1-distill-llama-70b" e
#              "llama-3.3-70b-versatile") foram descontinuados pela Groq e dao
#              erro 404; agora o modelo vem do seletor. E a busca usa a
#              DSADuckDuckGo, porque a DuckDuckGo do phidata devolve vazio.
# ==============================================================================
dsa_agente_web_search = Agent(name="DSA Agente Web Search",
                              role="Fazer busca na web",
                              model=dsa_cria_modelo(),
                              tools=[DSADuckDuckGo()],
                              instructions=["Sempre inclua as fontes"],
                              show_tool_calls=True, markdown=True)

dsa_agente_financeiro = Agent(name="DSA Agente Financeiro",
                              model=dsa_cria_modelo(),
                              tools=[YFinanceTools(stock_price=True,
                                                   analyst_recommendations=True,
                                                   stock_fundamentals=True,
                                                   company_news=True)],
                              instructions=["Use tabelas para mostrar os dados"],
                              show_tool_calls=True, markdown=True)

multi_ai_agent = Agent(team=[dsa_agente_web_search, dsa_agente_financeiro],
                       model=dsa_cria_modelo(),
                       instructions=["Sempre inclua as fontes", "Use tabelas para mostrar os dados"],
                       show_tool_calls=True, markdown=True)

print(f"3 agentes criados usando {DSA_PROVEDOR} / {DSA_MODELO}")
print(f"Time coordena {len(multi_ai_agent.team)} especialistas:",
      [a.name for a in multi_ai_agent.team])

## 2.6 Rodando o time de agentes

Esta é a chamada que o app faz quando você clica em **Analisar**.

A resposta do phidata vem com um bloco `Running:` no meio, listando as chamadas
de ferramenta e as transferências de tarefa entre agentes. Isso ajuda a depurar,
mas polui a resposta final — por isso o app limpa com uma expressão regular.

> **Detalhe que passou despercebido no material original:** o regex do curso
> procurava por `transfer_task_to_finance_ai_agent`, um nome que os agentes
> deste capítulo **nunca geram**. O nome real é derivado do `name` do agente,
> ou seja, `transfer_task_to_dsa_agente_financeiro`. Na versão modernizada o
> padrão aceita qualquer nome depois de `transfer_task_to_`.

In [ ]:
import re
import time
from IPython.display import Markdown

# Acima deste tempo NAO adianta esperar dentro do notebook (2 minutos).
DSA_ESPERA_MAXIMA = 120.0


def dsa_segundos_de_espera(mensagem_de_erro, padrao=15.0):
    """Le quantos segundos o provedor pediu para esperar antes de tentar de novo.

    A mensagem vem em formatos diferentes: "try again in 6.31s", "in 1m30s" e
    ate "in 4h30m36.288s" (quando o limite estourado e o DIARIO, nao o por
    minuto). Ler so os segundos, como se fosse sempre o primeiro formato, faz o
    programa esperar 15 segundos quando deveria esperar horas -- ele tenta de
    novo, falha de novo, e voce conclui que o codigo esta quebrado quando o
    problema era so a cota.
    """
    achado = re.search(r"try again in (?:(\d+)h)?(?:(\d+)m)?([\d.]+)s", mensagem_de_erro)
    if not achado:
        return padrao
    horas = int(achado.group(1) or 0)
    minutos = int(achado.group(2) or 0)
    segundos = float(achado.group(3) or 0)
    return horas * 3600 + minutos * 60 + segundos


def dsa_run_resiliente(agente, pergunta, tentativas=4):
    """Roda um agente repetindo apenas os erros que passam sozinhos.

    Dois erros aqui NAO sao bugs do seu codigo -- sao fatos da vida com LLMs:

    1) LIMITE DE USO (429). Um time de agentes conversa varias vezes com o
       modelo para responder uma unica pergunta, entao gasta muitos tokens.
       Sao dois limites diferentes e a diferenca importa: o POR MINUTO (TPM)
       libera em segundos e vale a pena esperar; o DIARIO (TPD) pode pedir
       horas -- ai a saida e trocar de provedor, nao esperar.

    2) FERRAMENTA INVENTADA (tool_use_failed). De vez em quando o modelo pede
       uma ferramenta que nao existe -- por exemplo "duckduckgo_open", que ele
       conhece de outros contextos. O provedor recusa a requisicao inteira com
       erro 400. Como e sorteio, repetir quase sempre resolve.

    E como pedir uma informacao a um atendente muito rapido: as vezes ele fala
    rapido demais e erra o nome do formulario. Voce so pede de novo.
    """
    for tentativa in range(1, tentativas + 1):
        try:
            return agente.run(pergunta)
        except Exception as erro:
            texto = str(erro)
            eh_limite = ("rate_limit" in texto) or ("429" in texto)
            eh_ferramenta = ("tool_use_failed" in texto) or ("was not in request.tools" in texto)
            if not (eh_limite or eh_ferramenta) or tentativa == tentativas:
                raise
            if eh_ferramenta:
                print(f"  [aviso] o modelo pediu uma ferramenta inexistente "
                      f"(tentativa {tentativa}/{tentativas}). Repetindo...")
                continue
            espera = dsa_segundos_de_espera(texto)
            if espera > DSA_ESPERA_MAXIMA:
                print(f"  [erro] cota DIARIA esgotada: o provedor pediu "
                      f"{espera/3600:.1f}h de espera. Troque de provedor no .env "
                      f"ou use o Ollama local.")
                raise
            print(f"  [aviso] limite por minuto atingido. Aguardando {espera:.0f}s "
                  f"(tentativa {tentativa}/{tentativas})...")
            time.sleep(espera)


ticker = "MSFT"

resposta = dsa_run_resiliente(
    multi_ai_agent,
    f"Resumir a recomendação do analista e compartilhar as últimas notícias para {ticker}"
)

print("Tipo do retorno:", type(resposta).__name__)
print("Tamanho bruto  :", len(resposta.content), "caracteres")
print()
print("--- RESPOSTA BRUTA (primeiros 400 caracteres) ---")
print(resposta.content[:400])

In [ ]:
# ==============================================================================
# CODIGO ORIGINAL (mantido comentado para referencia)
# ==============================================================================
# clean_response = re.sub(r"(Running:[\s\S]*?\n\n)|(^transfer_task_to_finance_ai_agent.*\n?)","", ai_response.content, flags=re.MULTILINE).strip()
#
# ==============================================================================
# VERSAO MODERNA (Python 3.12 + uv)
# O que mudou: o nome fixo "transfer_task_to_finance_ai_agent" nunca aparece de
#              verdade -- o phidata monta o nome a partir do "name" do agente.
#              Agora aceitamos qualquer "transfer_task_to_<qualquer coisa>".
# ==============================================================================
clean_response = re.sub(
    r"(Running:[\s\S]*?\n\n)|(^\s*-?\s*transfer_task_to_\w+.*\n?)",
    "",
    resposta.content,
    flags=re.MULTILINE,
).strip()

print(f"Antes: {len(resposta.content)} chars  ->  Depois: {len(clean_response)} chars")
print()
display(Markdown(clean_response))

---
# Parte 3 — Rodando o app completo

Com todas as peças testadas, o app é um comando só. Ele abre no navegador em
`http://localhost:8501`.

O `uv run` é o detalhe que evita a dor de cabeça mais comum: ele garante que o
Streamlit rode **com a `.venv` deste capítulo**, sem você precisar ativar nada
na mão.

In [ ]:
# ============================================================================
# RODAR O APP  (rode no TERMINAL, nao aqui)
# ============================================================================
# Um notebook nao e o lugar de rodar um servidor web -- ele ficaria travado.

# --- Local ---
# uv run streamlit run dsa_app.py

# --- Se a porta 8501 ja estiver ocupada ---
# uv run streamlit run dsa_app.py --server.port=8502

# --- Na AWS EC2 (como mostrado nas aulas) ---
# uv run streamlit run dsa_app.py --server.port=8501 --server.address=0.0.0.0
# Em segundo plano, para nao morrer quando voce fechar o terminal:
# nohup uv run streamlit run dsa_app.py --server.port=8501 --server.address=0.0.0.0 &

# --- Ver os modelos validos da sua conta (quando der erro 404 de modelo) ---
# uv run python dsa_lista_modelos.py

---
# Parte 4 — Trocando o motor: Groq, OpenAI, Anthropic, Grok e Ollama

Este é o ponto pedagógico central da modernização.

Todo o app conversa com o LLM através de **um único objeto**, aquele que você
passa em `model=`. Nada mais no código sabe quem está do outro lado. Trocar de
provedor é trocar esse objeto — e mais nada.

É como a tomada da parede: o liquidificador não sabe se a energia veio de uma
hidrelétrica ou de um painel solar. Ele só precisa que o formato do plugue
combine. O `model=` é o plugue.

Vamos ver isso de duas formas:

- **4.1 a 4.3** — com o phidata (o jeito do capítulo): troque o objeto, o resto fica igual;
- **4.4 em diante** — sem framework nenhum, falando direto com o SDK de cada
  provedor, para você enxergar o que o phidata faz por baixo do capô.

## 4.1 Quais modelos a sua conta realmente tem?

**Esta é a célula que resolve o erro 404 de nome de modelo** — o erro mais comum
neste capítulo, porque os provedores aposentam modelos com frequência.

Provedor sem chave configurada é **pulado sem erro**.

In [ ]:
def dsa_lista_modelos():
    # --- GROQ ---
    print("=" * 70); print("GROQ"); print("=" * 70)
    chave = dsa_chave_valida("GROQ_API_KEY")
    if not chave:
        print("  sem chave no .env -- pulado")
    else:
        from groq import Groq as GroqSDK
        ids = sorted(m.id for m in GroqSDK(api_key=chave).models.list().data)
        for i in ids:
            print("  -", i)
        for antigo in ["deepseek-r1-distill-llama-70b", "llama-3.3-70b-versatile"]:
            situacao = "AINDA EXISTE" if antigo in ids else "DESCONTINUADO (erro 404)"
            print(f"  >> modelo do video '{antigo}': {situacao}")

    # --- OPENAI ---
    print("\n" + "=" * 70); print("OPENAI"); print("=" * 70)
    chave = dsa_chave_valida("OPENAI_API_KEY")
    if not chave:
        print("  sem chave no .env -- pulado")
    else:
        from openai import OpenAI
        ids = sorted(m.id for m in OpenAI(api_key=chave).models.list().data)
        conversa = [i for i in ids if i.startswith(("gpt-", "o1", "o3", "o4"))]
        print(f"  {len(conversa)} modelos de conversa (de {len(ids)} no total):")
        for i in conversa[:25]:
            print("  -", i)

    # --- ANTHROPIC ---
    print("\n" + "=" * 70); print("ANTHROPIC"); print("=" * 70)
    chave = dsa_chave_valida("ANTHROPIC_API_KEY")
    if not chave:
        print("  sem chave no .env -- pulado")
    else:
        import anthropic
        for m in anthropic.Anthropic(api_key=chave).models.list().data:
            print("  -", m.id)

    # --- GROK (xAI) ---
    print("\n" + "=" * 70); print("GROK (xAI)"); print("=" * 70)
    chave = dsa_chave_valida("XAI_API_KEY")
    if not chave:
        print("  sem chave no .env -- pulado")
    else:
        from openai import OpenAI
        cliente = OpenAI(api_key=chave, base_url="https://api.x.ai/v1")
        for m in sorted(x.id for x in cliente.models.list().data):
            print("  -", m)

    # --- OLLAMA ---
    print("\n" + "=" * 70); print("OLLAMA (local, sem chave)"); print("=" * 70)
    if OLLAMA_HOST:
        for m in MODELOS_LOCAIS:
            print("  -", m)
    else:
        print("  servidor local nao encontrado")


dsa_lista_modelos()

## 4.2 Teste de conexão, provedor por provedor

Antes de comparar respostas, vale confirmar que cada provedor **responde**. Esta
célula faz a pergunta mais barata possível ("responda apenas: OK") em cada um.

Se um provedor falhar aqui, o problema é de chave ou de nome de modelo — não do
código do capítulo.

In [ ]:
import time

def dsa_testa_conexao():
    resultados = []

    def tenta(nome, funcao):
        inicio = time.time()
        try:
            texto = funcao()
            resultados.append((nome, "OK", f"{time.time()-inicio:.1f}s", texto[:40]))
        except Exception as erro:
            resultados.append((nome, "FALHOU", "-", f"{type(erro).__name__}: {erro}"[:70]))

    PERGUNTA = [{"role": "user", "content": "Responda apenas: OK"}]

    if dsa_chave_valida("GROQ_API_KEY"):
        from groq import Groq as GroqSDK
        tenta("Groq", lambda: GroqSDK(api_key=dsa_chave_valida("GROQ_API_KEY"))
              .chat.completions.create(model=os.getenv("GROQ_MODEL") or "openai/gpt-oss-120b",
                                       messages=PERGUNTA).choices[0].message.content)
    else:
        resultados.append(("Groq", "sem chave", "-", "provedor pulado"))

    if dsa_chave_valida("OPENAI_API_KEY"):
        from openai import OpenAI
        tenta("OpenAI", lambda: OpenAI(api_key=dsa_chave_valida("OPENAI_API_KEY"))
              .chat.completions.create(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini",
                                       messages=PERGUNTA).choices[0].message.content)
    else:
        resultados.append(("OpenAI", "sem chave", "-", "provedor pulado"))

    if dsa_chave_valida("ANTHROPIC_API_KEY"):
        import anthropic
        tenta("Anthropic", lambda: anthropic.Anthropic(api_key=dsa_chave_valida("ANTHROPIC_API_KEY"))
              .messages.create(model=os.getenv("ANTHROPIC_MODEL") or "claude-sonnet-4-5",
                               max_tokens=64, messages=PERGUNTA).content[0].text)
    else:
        resultados.append(("Anthropic", "sem chave", "-", "provedor pulado"))

    if dsa_chave_valida("XAI_API_KEY"):
        from openai import OpenAI
        tenta("Grok", lambda: OpenAI(api_key=dsa_chave_valida("XAI_API_KEY"),
                                     base_url="https://api.x.ai/v1")
              .chat.completions.create(model=os.getenv("XAI_MODEL") or "grok-4",
                                       messages=PERGUNTA).choices[0].message.content)
    else:
        resultados.append(("Grok", "sem chave", "-", "provedor pulado"))

    if OLLAMA_HOST:
        import ollama
        tenta("Ollama", lambda: ollama.Client(host=OLLAMA_HOST, timeout=120)
              .chat(model=os.getenv("OLLAMA_MODEL") or "llama3.2",
                    messages=PERGUNTA).message.content)
    else:
        resultados.append(("Ollama", "sem servidor", "-", "provedor pulado"))

    print(f"{'PROVEDOR':<12} {'STATUS':<12} {'TEMPO':<8} DETALHE")
    print("-" * 78)
    for linha in resultados:
        print(f"{linha[0]:<12} {linha[1]:<12} {linha[2]:<8} {linha[3]}")


dsa_testa_conexao()

## 4.3 O mesmo agente, motores diferentes

Aqui a ideia fica visível. O código do agente é **escrito uma vez** e roda com
qualquer provedor. A única linha que muda é a do `model=`.

In [ ]:
def dsa_agente_com(provedor):
    """Monta o MESMO agente financeiro, mudando so o motor."""
    if provedor == "Groq":
        motor = Groq(id=os.getenv("GROQ_MODEL") or "openai/gpt-oss-120b",
                     api_key=dsa_chave_valida("GROQ_API_KEY"))
    elif provedor == "OpenAI":
        motor = OpenAIChat(id=os.getenv("OPENAI_MODEL") or "gpt-4o-mini",
                           api_key=dsa_chave_valida("OPENAI_API_KEY"))
    elif provedor == "Anthropic":
        motor = Claude(id=os.getenv("ANTHROPIC_MODEL") or "claude-sonnet-4-5",
                       api_key=dsa_chave_valida("ANTHROPIC_API_KEY"))
    elif provedor == "Grok":
        motor = xAI(id=os.getenv("XAI_MODEL") or "grok-4",
                    api_key=dsa_chave_valida("XAI_API_KEY"))
    else:
        motor = Ollama(id=os.getenv("OLLAMA_MODEL") or "llama3.2", host=OLLAMA_HOST)

    # Daqui para baixo NADA depende do provedor. Este e o ponto.
    return Agent(name="DSA Agente Financeiro",
                 model=motor,
                 tools=[YFinanceTools(stock_price=True)],
                 instructions=["Responda em portugues, em uma frase"],
                 show_tool_calls=True, markdown=True)


DISPONIVEIS = [p for p, v, _, _ in DSA_PROVEDORES if dsa_chave_valida(v)]
if OLLAMA_HOST:
    DISPONIVEIS.append("Ollama")

print("Provedores disponiveis nesta maquina:", DISPONIVEIS)
print()

for provedor in DISPONIVEIS:
    try:
        r = dsa_run_resiliente(dsa_agente_com(provedor),
                               "Qual o preço atual da ação MSFT?")
        limpo = re.sub(r"(Running:[\s\S]*?\n\n)|(^\s*-?\s*\w+\(.*\)\s*$)", "",
                       r.content, flags=re.MULTILINE).strip()
        print(f"[{provedor}] {limpo[:150]}")
    except Exception as erro:
        print(f"[{provedor}] FALHOU: {type(erro).__name__}: {str(erro)[:120]}")
    print()

## 4.4 Sem framework: falando direto com cada provedor

O phidata esconde as diferenças entre provedores. Isso é ótimo no dia a dia, mas
atrapalha o aprendizado — parece mágica. Nesta seção falamos direto com cada SDK.

Três coisas que você vai perceber:

1. **OpenAI e Grok usam a mesma biblioteca.** Literalmente. A xAI copiou o
   formato da OpenAI, então você só troca o `base_url`. A Groq também segue esse
   formato.
2. **A Anthropic é a diferente.** A mensagem de sistema não vai dentro de
   `messages`, vai num parâmetro separado `system=`. E o JSON estruturado usa
   `output_config`, não `response_format`.
3. **Todos podem devolver JSON embrulhado em ```` ```json ````**, mesmo quando
   você pede JSON puro. Por isso existe uma função de limpeza.

Vamos construir um **adaptador**: mesma entrada, mesma saída, para todos.

In [ ]:
# ==============================================================================
# REDE DE SEGURANCA 1: objeto de resposta que funciona por chave E por atributo
# ==============================================================================
# Por que isso importa? O cliente do Ollama devolve um ChatResponse, que aceita
# tanto r["message"]["content"] quanto r.message.content. Se o nosso adaptador
# devolvesse um dicionario simples, o codigo que usa ponto quebraria. Esta
# classe e um dicionario que TAMBEM responde a ponto -- assim ela e um
# substituto direto para o objeto do Ollama.

class DSAResposta(dict):
    """Dicionario que tambem aceita acesso por atributo (r.texto == r['texto'])."""

    def __getattr__(self, nome):
        try:
            valor = self[nome]
        except KeyError:
            raise AttributeError(nome) from None
        return DSAResposta(valor) if isinstance(valor, dict) else valor


# ==============================================================================
# REDE DE SEGURANCA 2: limpar cercas de markdown em volta do JSON
# ==============================================================================
# Voce pede "devolva JSON" e o modelo responde:
#
#     Claro! Aqui esta:
#     ```json
#     {"acao": "MSFT"}
#     ```
#
# O json.loads engasga com isso. Esta funcao descasca o embrulho.

def dsa_limpa_json(texto):
    """Remove cercas ```json e texto em volta, devolvendo so o JSON."""
    if not texto:
        return texto
    t = texto.strip()
    if "```" in t:
        # Pega o conteudo do primeiro bloco cercado
        partes = re.split(r"```(?:json)?", t)
        if len(partes) >= 2:
            t = partes[1].strip()
    # Se ainda sobrou texto em volta, corta do primeiro { ou [ ate o fecho
    inicio = min([i for i in (t.find("{"), t.find("[")) if i != -1], default=-1)
    if inicio > 0:
        fim = max(t.rfind("}"), t.rfind("]"))
        if fim > inicio:
            t = t[inicio:fim + 1]
    return t.strip()


# Teste rapido da limpeza
exemplo = 'Claro! Aqui esta:\n```json\n{"acao": "MSFT", "preco": 493.95}\n```\nEspero ter ajudado!'
print("Texto sujo :", repr(exemplo[:50]), "...")
print("Depois     :", dsa_limpa_json(exemplo))
print("json.loads :", json.loads(dsa_limpa_json(exemplo)))

### Os adaptadores

Todos têm a **mesma assinatura**: recebem uma lista de mensagens e devolvem um
`DSAResposta` com o campo `.texto`. Quem chama não precisa saber qual provedor
está do outro lado.

In [ ]:
def dsa_chama_openai(mensagens, modelo=None, pedir_json=False, chave=None, base_url=None):
    """Adaptador OpenAI -- e tambem o do Grok e o da Groq, que copiam esse formato."""
    from openai import OpenAI

    cliente = OpenAI(api_key=chave or dsa_chave_valida("OPENAI_API_KEY"), base_url=base_url)
    extras = {}
    if pedir_json:
        # ATENCAO: com response_format json_object, a palavra "json" precisa
        # aparecer em ALGUMA das mensagens, senao a API recusa a chamada.
        extras["response_format"] = {"type": "json_object"}
    r = cliente.chat.completions.create(
        model=modelo or os.getenv("OPENAI_MODEL") or "gpt-4o-mini",
        messages=mensagens,
        **extras,
    )
    return DSAResposta(texto=r.choices[0].message.content,
                       modelo=r.model,
                       provedor="OpenAI" if not base_url else "compativel-OpenAI")


def dsa_chama_grok(mensagens, modelo=None, pedir_json=False):
    """Grok (xAI): MESMA biblioteca da OpenAI, so muda o endereco do servidor."""
    r = dsa_chama_openai(mensagens,
                         modelo=modelo or os.getenv("XAI_MODEL") or "grok-4",
                         pedir_json=pedir_json,
                         chave=dsa_chave_valida("XAI_API_KEY"),
                         base_url="https://api.x.ai/v1")
    r["provedor"] = "Grok"
    return r


def dsa_chama_groq(mensagens, modelo=None, pedir_json=False):
    """Groq: tambem segue o formato da OpenAI (mas tem biblioteca propria)."""
    from groq import Groq as GroqSDK

    cliente = GroqSDK(api_key=dsa_chave_valida("GROQ_API_KEY"))
    extras = {"response_format": {"type": "json_object"}} if pedir_json else {}
    r = cliente.chat.completions.create(
        model=modelo or os.getenv("GROQ_MODEL") or "openai/gpt-oss-120b",
        messages=mensagens,
        **extras,
    )
    return DSAResposta(texto=r.choices[0].message.content, modelo=r.model, provedor="Groq")


def dsa_chama_ollama(mensagens, modelo=None, pedir_json=False):
    """Ollama local: aceita format='json' para forcar JSON."""
    import ollama

    cliente = ollama.Client(host=OLLAMA_HOST or "http://localhost:11434", timeout=180)
    extras = {"format": "json"} if pedir_json else {}
    r = cliente.chat(model=modelo or os.getenv("OLLAMA_MODEL") or "llama3.2",
                     messages=mensagens, **extras)
    return DSAResposta(texto=r.message.content, modelo=modelo or "llama3.2", provedor="Ollama")

### O adaptador da Anthropic — o que é diferente

A Anthropic merece explicação separada porque tem três particularidades que
derrubam quem chega da OpenAI:

1. **A mensagem de sistema vai fora de `messages`.** Vai no parâmetro `system=`.
   Se você mandar `{"role": "system", ...}` dentro de `messages`, a API recusa.
2. **`max_tokens` é obrigatório.** Não tem valor padrão.
3. **Prefill de assistente não funciona mais.** A técnica antiga de começar a
   resposta com `{"role": "assistant", "content": "{"}` para forçar JSON
   **retorna erro 400** nos modelos atuais. Para JSON, use `output_config` com
   `json_schema`.

Vale também checar `stop_reason == "refusal"`: é como o modelo diz que se
recusou a responder, e o texto vem vazio.

> **Aviso honesto:** esta função **não pôde ser testada** durante a modernização
> deste capítulo, porque não havia chave da Anthropic disponível. Ela segue a
> documentação atual da API. Se você tiver uma chave, a célula 4.2 vai exercitá-la.

In [ ]:
def dsa_chama_anthropic(mensagens, modelo=None, pedir_json=False, esquema=None,
                        esforco_baixo=False):
    """Adaptador Anthropic (Claude).

    Diferencas em relacao a OpenAI, todas tratadas aqui dentro:
      - a mensagem de sistema sai de 'messages' e vai para o parametro system=
      - max_tokens e obrigatorio
      - JSON estruturado usa output_config com json_schema (nao response_format)
    """
    import anthropic

    cliente = anthropic.Anthropic(api_key=dsa_chave_valida("ANTHROPIC_API_KEY"))

    # Separa a mensagem de sistema das demais -- exigencia da API.
    sistema = " ".join(m["content"] for m in mensagens if m.get("role") == "system")
    conversa = [m for m in mensagens if m.get("role") != "system"]

    parametros = {
        "model": modelo or os.getenv("ANTHROPIC_MODEL") or "claude-sonnet-4-5",
        "max_tokens": 1024,          # obrigatorio na Anthropic
        "messages": conversa,
    }
    if sistema:
        parametros["system"] = sistema

    output_config = {}
    if pedir_json:
        # NAO use prefill de assistente ({"role":"assistant","content":"{"}):
        # foi removido nos modelos atuais e retorna erro 400.
        output_config["format"] = {
            "type": "json_schema",
            "schema": esquema or {
                "type": "object",
                "properties": {"resposta": {"type": "string"}},
                "required": ["resposta"],
            },
        }
    if esforco_baixo:
        # Reduz custo e latencia. ATENCAO: o Haiku NAO aceita este parametro.
        output_config["effort"] = "low"
    if output_config:
        parametros["output_config"] = output_config

    r = cliente.messages.create(**parametros)

    # O modelo pode se recusar a responder; nesse caso o texto vem vazio.
    if getattr(r, "stop_reason", None) == "refusal":
        return DSAResposta(texto="", provedor="Anthropic", modelo=r.model,
                           recusado=True)

    texto = "".join(bloco.text for bloco in r.content if getattr(bloco, "type", "") == "text")
    return DSAResposta(texto=texto, provedor="Anthropic", modelo=r.model, recusado=False)


print("Adaptadores prontos: OpenAI, Grok, Groq, Ollama, Anthropic")
print("Todos recebem a mesma lista de mensagens e devolvem .texto")

## 4.5 Comparativo lado a lado

A mesma pergunta, para todos os provedores configurados. Repare que a **chamada
é idêntica** — muda só qual função do adaptador é usada.

Provedores sem chave são pulados sem erro.

In [ ]:
PERGUNTA = [
    {"role": "system", "content": "Você é um analista financeiro objetivo. Responda em português."},
    {"role": "user", "content": "Em no máximo 2 frases: o que é 'day trade' e qual o principal risco?"},
]

ADAPTADORES = [
    ("Groq",      dsa_chama_groq,      lambda: dsa_chave_valida("GROQ_API_KEY")),
    ("OpenAI",    dsa_chama_openai,    lambda: dsa_chave_valida("OPENAI_API_KEY")),
    ("Anthropic", dsa_chama_anthropic, lambda: dsa_chave_valida("ANTHROPIC_API_KEY")),
    ("Grok",      dsa_chama_grok,      lambda: dsa_chave_valida("XAI_API_KEY")),
    ("Ollama",    dsa_chama_ollama,    lambda: OLLAMA_HOST),
]

for nome, funcao, disponivel in ADAPTADORES:
    print("=" * 78)
    if not disponivel():
        print(f"{nome}: nao configurado -- pulado (sem erro)")
        print()
        continue
    try:
        inicio = time.time()
        r = funcao(PERGUNTA)
        # Prova de que o objeto funciona das duas formas:
        assert r.texto == r["texto"]
        print(f"{nome}  ({r.modelo}, {time.time()-inicio:.1f}s)")
        print("-" * 78)
        print(r.texto.strip()[:600])
    except Exception as erro:
        print(f"{nome}: FALHOU -- {type(erro).__name__}: {str(erro)[:200]}")
    print()

### E o JSON estruturado?

Quando você precisa que a resposta seja **dados**, e não texto corrido, peça
JSON. Cada provedor tem seu jeito, e o adaptador já esconde isso — mas a rede de
segurança (`dsa_limpa_json`) continua valendo, porque nenhum modelo é 100%
obediente.

In [ ]:
PEDIDO_JSON = [
    {"role": "system", "content": "Você devolve apenas json válido, sem explicação."},
    {"role": "user", "content": 'Devolva um json com as chaves "ticker" e "empresa" para MSFT.'},
]

for nome, funcao, disponivel in ADAPTADORES:
    if not disponivel():
        print(f"{nome}: nao configurado -- pulado")
        continue
    try:
        r = funcao(PEDIDO_JSON, pedir_json=True)
        limpo = dsa_limpa_json(r.texto)
        dados = json.loads(limpo)
        print(f"{nome:<10} OK   -> {dados}")
    except Exception as erro:
        print(f"{nome:<10} FALHOU -> {type(erro).__name__}: {str(erro)[:120]}")

---
# Parte 5 — Rotina e problemas comuns

## 5.1 Como iniciar o ambiente nas próximas vezes

Você já fez a instalação. No dia a dia, a rotina é curta — três passos.

In [ ]:
# ============================================================================
# ROTINA DO DIA A DIA  (rode no TERMINAL)
# ============================================================================

# 1) Abrir o VSCode NA PASTA DO CAPITULO (isto importa: veja o LEIAME.txt)
# cd ~/projects/dsa/12-Projeto
# code .

# 2) Conferir se o ambiente esta ok (so na primeira vez do dia, e rapido)
# uv sync

# 3a) Para usar o NOTEBOOK: abra o .ipynb e escolha o kernel
#     Select Kernel > Python Environments... > .venv (Python 3.12)
#     Confira sempre a celula 1.5: o sys.executable tem que apontar para a .venv

# 3b) Para usar o APP:
# uv run streamlit run dsa_app.py

# --- Se o Ollama nao estiver no ar (so quando nao houver systemd) ---
# ollama serve

# --- Alternativa ao VSCode, se preferir o Jupyter no navegador ---
# uv run jupyter lab

## 5.2 Problemas comuns

| Sintoma | Causa provável | Solução |
|---|---|---|
| `404 model_not_found` / `The model ... does not exist` | O modelo foi descontinuado pelo provedor. Aconteceu com `deepseek-r1-distill-llama-70b` e `llama-3.3-70b-versatile` | Rode a célula 4.1 (ou `uv run python dsa_lista_modelos.py`), escolha um nome válido e ajuste `GROQ_MODEL` no `.env` |
| O agente de busca nunca acha nada, mas não dá erro | `duckduckgo_search` virou casca vazia; foi renomeado para `ddgs` | Use a classe `DSADuckDuckGo` (célula 2.4), que já vem no `dsa_app.py` |
| `API key not set` mesmo com o `.env` preenchido | `load_dotenv()` sem argumento procura o `.env` ao lado do **script**, não na pasta atual | Use `load_dotenv(Path(__file__).parent / ".env")` |
| `ModuleNotFoundError` para qualquer pacote | O kernel selecionado não é o da `.venv` | Célula 1.5: confira o `sys.executable`. Depois Select Kernel → `.venv (Python 3.12)` |
| Botão "Select Kernel" não aparece no VSCode | Faltam as extensões dentro do WSL | `code --install-extension ms-python.python` e depois `code --install-extension ms-toolsai.jupyter` (uma de cada vez), então Reload Window |
| `ERROR: This version requires zstd for extraction` | Instalador do Ollama sem o `zstd` | `sudo apt-get install -y zstd` e reinstale |
| Ollama não responde em `localhost` no WSL2 | O Ollama está instalado no **Windows**, não no Linux | Célula 1.6 detecta sozinha; ou defina `OLLAMA_HOST` com o IP do `nameserver` do `/etc/resolv.conf` |
| Gráficos aparecem espremidos no app | `st.plotly_chart(fig)` usa o tamanho nativo | `st.plotly_chart(fig, width="stretch")` — o antigo `use_container_width=True` está deprecated |
| Tabela vazia e gráficos em branco | Ticker inexistente. O `yfinance` não levanta erro, devolve tabela vazia | O app já checa com `if hist.empty`. Use tickers da Nasdaq: MSFT, TSLA, AMZN, GOOG |
| `429` dizendo `tokens per day (TPD)` e pedindo horas de espera | A cota **diária** do provedor acabou. Repetir não resolve | Use outro provedor (basta a chave no `.env`), o Ollama local (sem cota), ou aumente o plano |
| `429` dizendo `tokens per minute (TPM)` | Um time de agentes gasta muitos tokens por pergunta | Espere um minuto. O `dsa_run_resiliente` e o `dsa_app.py` já esperam o tempo que o provedor pedir |
| `tool_use_failed` / `was not in request.tools` | O modelo pediu uma ferramenta que não existe (ex.: `duckduckgo_open`). Acontece por variação de amostragem, não é bug do seu código | Repetir. O `dsa_run_resiliente` e o `dsa_app.py` já fazem isso sozinhos, até 3 vezes |
| `json.loads` falha na resposta do LLM | O modelo embrulhou o JSON em ```` ```json ```` | Use `dsa_limpa_json()` antes do `json.loads` |
| Erro 400 na Anthropic ao forçar JSON | Prefill de assistente foi removido nos modelos atuais | Use `output_config={"format": {"type": "json_schema", ...}}` |
| Porta 8501 ocupada | Outro Streamlit já rodando | `uv run streamlit run dsa_app.py --server.port=8502` |

---

## Fim

Você tem agora o capítulo rodando em ambiente moderno, com o código original
preservado comentado para comparação, e com liberdade para escolher o motor de
IA — inclusive um gratuito, rodando na sua própria máquina.

**Obrigado DSA!**